In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# La₀.₅Ba₀.₅CoO₃ on HRPT

This tutorial fits the cubic perovskite La₀.₅Ba₀.₅CoO₃ to constant-wavelength neutron powder data
measured on HRPT at PSI. It builds the structure by hand, loads the measured pattern from
EasyDiffraction's collection, describes the experiment, links the two and fits them, showing the
pattern and the parameters after each step. The other tutorials load their structures from a CIF.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

## Import the library

In [ ]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [ ]:
project = edi.Project(name='lbco_hrpt')

## Define the structure

La and Ba share the corner site of the cubic cell, Co sits at its centre and O on the face
centres, in space group P m -3 m. Every atom starts with an isotropic displacement of 0.5 Å².

In [ ]:
project.structures.create(name='lbco')
structure = project.structures['lbco']
structure.space_group.name_h_m = 'P m -3 m'
structure.cell.length_a = 3.88

In [ ]:
structure.atom_sites.create(
    id='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    occupancy=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    occupancy=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_iso=0.5,
)

In [ ]:
project.display.structure('lbco')

## Define the experiment

The measured pattern comes from EasyDiffraction's collection: 2θ, the counts and their standard
uncertainties.

In [ ]:
data_path = edi.download_data('pd-neut-cwl_lbco-hrpt-hrpt-data', destination='data')
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['hrpt']

HRPT's wavelength is 1.494 Å. The 2θ zero starts at 0.3°: from 0 the low-angle peaks sit more
than a peak width away and the fit settles in a wrong minimum. The profile is the
Thompson-Cox-Hastings pseudo-Voigt, its terms to one significant figure.

In [ ]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.3
experiment.peak.broad_gauss_u = 0.08
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1
experiment.peak.broad_lorentz_y = 0.08

A flat background of five points, and the two ends of the pattern left out of the fit.

In [ ]:
for position in (10, 30, 50, 110, 165):
    experiment.background.create(position=position, intensity=160)
experiment.excluded_regions.create(start=0, end=5)
experiment.excluded_regions.create(start=165, end=180)

The experiment shows the structure it links, here `lbco` with a scale of 9.

In [ ]:
experiment.linked_structures.create(structure_id='lbco', scale=9)

In [ ]:
project.display.pattern('hrpt')

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time, so each stage starts close to
its answer.

### Scale, 2θ zero and cell

The scale sets the overall height of the pattern, and the zero shift and the cell length put the
peaks where they are measured, so they come first.

In [ ]:
experiment.linked_structures['lbco'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('hrpt')

### Background

With the peaks in place, the five background points follow the level between them.

In [ ]:
for point in experiment.background:
    point.intensity.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('hrpt')

### Peak widths

The profile terms now fit the widths and shapes of the peaks.

In [ ]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('hrpt')

### Displacements

Every atom sits on a special position of P m -3 m, so there are no coordinates to fit. The
displacements change the peak intensities with angle and come last.

In [ ]:
structure.atom_sites['La'].adp_iso.free = True
structure.atom_sites['Ba'].adp_iso.free = True
structure.atom_sites['Co'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('hrpt')

In [ ]:
project.display.fit.correlations()

### Constrain La and Ba

The correlations show the displacements of La and Ba moving together almost exactly, with
opposite signs: they share one site and scatter neutrons similarly, so the pattern cannot tell
them apart, and their s.u. are large. Give each an alias and constrain Ba's to follow La's, then
fit again. Ba's displacement is no longer fitted on its own.

In [ ]:
structure.atom_sites['Ba'].adp_iso.free = False
project.analysis.aliases.create(id='biso_La', param=structure.atom_sites['La'].adp_iso)
project.analysis.aliases.create(id='biso_Ba', param=structure.atom_sites['Ba'].adp_iso)
project.analysis.constraints.create(id='biso_Ba', expression='biso_Ba = biso_La')
project.display.constraints()

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('hrpt')

In [ ]:
project.display.pattern('hrpt', x_min=38, x_max=41)

The pair of La and Ba has gone from the correlations, and La's displacement now has a small s.u.

In [ ]:
project.display.fit.correlations()

## Save the project

Save the fitted project, and its report as one HTML file that opens without a network
connection.

In [ ]:
project.save_as('lbco_hrpt')
project.report.save_html('lbco_hrpt-report.html', offline=True)